In [1]:
!pip install ragas[all]
!pip install litellm

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 3.3 MB/s eta 0:00:00
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 161.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 19.3 MB/s eta 0:00:00
  Attempting uninstall: importlib-metadata
    Found existing installation: importlib-metadata 4.6.4
    Uninstalling importlib-metadata-4.6.4:
      Successfully uninstalled importlib-metadata-4.6.4


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:

# Create a persistent models directory in Google Drive
!mkdir -p /content/drive/MyDrive/RAG/ollama_models

# Remove any existing ephemeral models directory
!rm -rf ~/.ollama/models

# Symlink Ollama's model directory to Google Drive
!ln -s /content/drive/MyDrive/RAG/ollama_models ~/.ollama/models

print('Symlinked ~/.ollama/models -> /content/drive/MyDrive/RAG/ollama_models')

import subprocess, time

server = subprocess.Popen(['ollama', 'serve'])
print('Ollama server started as background process (PID:', server.pid, ')')
time.sleep(5)
print('Waited 5 seconds for the server to initialize.')



ln: failed to create symbolic link '/root/.ollama/models': No such file or directory
Symlinked ~/.ollama/models -> /content/drive/MyDrive/RAG/ollama_models
Ollama server started as background process (PID: 6705 )
Waited 5 seconds for the server to initialize.


In [4]:
# LLM
!ollama pull gpt-oss:20b

!ollama pull nomic-embed-text

In [5]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory

aclient = AsyncOpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

llm = llm_factory(
    provider="openai",
    model="gpt-oss:20b",
    client=aclient,
    temperature=0,
    max_tokens=100000,
)

embeddings = embedding_factory(
    provider="openai",
    model="nomic-embed-text",
    client=aclient,
)


/usr/lib/python3/dist-packages/blinker/base.py:96: SyntaxWarning: invalid escape sequence '\*'
  `sender=` as a single positional argument and any \*\*kwargs that
/usr/lib/python3/dist-packages/blinker/base.py:174: SyntaxWarning: invalid escape sequence '\*'
  `sender=` as a single positional argument and any \*\*kwargs that
/usr/lib/python3/dist-packages/blinker/base.py:242: SyntaxWarning: invalid escape sequence '\*'
  """Emit this signal on behalf of *sender*, passing on \*\*kwargs.
/tmp/ipython-input-3316367304.py:18: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings = embedding_factory(


In [8]:
!pip install openpyxl
import pandas as pd

eval_dataset = pd.read_excel('/content/drive/MyDrive/RAG/datasets/adv_rag_eval_dataset.xlsx', sheet_name="adv_llama")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 9.7 MB/s eta 0:00:00


In [9]:
# Define a dummy Document class to handle the eval of the string representation
class Document:
    def __init__(self, page_content=None, metadata=None, **kwargs):
        self.page_content = page_content
        self.metadata = metadata
        self.id = kwargs.get('id')

    def __repr__(self):
        return f"Document(page_content='{self.page_content}')"

def parse_sub_questions(row):
    # Initialize default values for up to 3 sub-questions
    result = {}
    for i in range(1, 4):
        result[f'subquestion_{i}'] = None
        result[f'query_{i}'] = None
        result[f'page_content_{i}'] = None

    if pd.isna(row):
        return pd.Series(result)

    try:
        # Evaluate the string to a dictionary, using the dummy Document class
        data = eval(row, {"Document": Document})

        for i in range(1, 4):
            key = f'sub_question_{i}'
            if key in data:
                sub_q = data[key]
                # Mapping extracted fields to requested columns
                # subquestion_i -> mapped to the 'answer' of the sub-question
                # query_i -> mapped to the 'query' of the sub-question
                result[f'subquestion_{i}'] = sub_q.get('answer')
                result[f'query_{i}'] = sub_q.get('query')

                # Extract page content from contexts
                contexts = sub_q.get('contexts', [])
                if isinstance(contexts, list):
                    contents = [doc.page_content for doc in contexts if hasattr(doc, 'page_content')]
                    result[f'page_content_{i}'] = "\n\n".join(contents)
    except Exception as e:
        # In case of parsing errors, return empty results for this row
        pass

    return pd.Series(result)

# Apply the parsing function to the dataframe
expanded_cols = eval_dataset['retrieved_contexts'].apply(parse_sub_questions)

# Concatenate with original relevant columns
# We include 'index' by resetting the index, and keep 'user_input' and 'answer'
eval_dataset_df = pd.concat([eval_dataset[['id', 'user_input', 'answer', 'answer_ref', 'reference']], expanded_cols], axis=1)
eval_dataset_df = eval_dataset_df.reset_index()

# Select and reorder columns as requested
cols = ['index', 'user_input', 'answer', 'answer_ref', 'reference']
for i in range(1, 4):
    cols.extend([f'subquestion_{i}', f'query_{i}', f'page_content_{i}'])

# Create the final dataframe
eval_dataset_df = eval_dataset_df[cols]

# Safely join page_content_1/2/3 into a single string column `retrieved_context`
def safe_join_page_contents(row):
    cols = ['page_content_1', 'page_content_2', 'page_content_3']
    parts = []
    for c in cols:
        v = row.get(c) if isinstance(row, dict) else row[c]
        if v is None:
            continue
        if isinstance(v, float) and pd.isna(v):
            continue
        s = str(v).strip()
        if s and s.lower() != 'nan':
            parts.append(s)
    # join with double newlines so different contexts remain visually separated
    return "\n\n".join(parts)

# Create a single text column
eval_dataset_df['retrieved_context'] = eval_dataset_df.apply(safe_join_page_contents, axis=1)

# Also create a list-style column (useful for evaluation libraries that expect lists)
eval_dataset_df['retrieved_contexts'] = eval_dataset_df['retrieved_context'].apply(lambda s: [s] if s else [])

display(eval_dataset_df.head())

,index,user_input,answer,answer_ref,reference,subquestion_1,query_1,page_content_1,subquestion_2,query_2,page_content_2,subquestion_3,query_3,page_content_3,retrieved_context,retrieved_contexts
0,0,Which month is set aside each year in the UAE ...,"Based on the information provided, the final a...",March,['Building on the Abu Dhabi Reads campaign lau...,"According to the documents, building on the Ab...",What are the nationwide reading and literacy a...,international reading performance benchmark fo...,"Yes, according to the text, the month of March...",Is there a specific month set aside each year ...,international reading performance benchmark fo...,None,None,None,international reading performance benchmark fo...,[international reading performance benchmark f...
1,1,"How many public, private, and charter schools ...","Based on the provided documents, the total num...","Abu Dhabi has 218 public schools, 203 private ...",['There are 218 public schools with a student ...,"Based on the provided documents, there are thr...","What are the types of schools (public, private...",There are 218 public schools with a student po...,"According to the documents, there are **218** ...",How many public schools exist in Abu Dhabi?,There are 218 public schools with a student po...,"According to the provided documents, there are...",How many private and charter schools exist in ...,There are 218 public schools with a student po...,There are 218 public schools with a student po...,[There are 218 public schools with a student p...
2,2,What mandatory early education requirement mus...,"In Austria, there is no specific age or year r...",Austrian children must complete one compulsory...,"['Since 2010, one year of Kindergarten has bee...","According to the text, children whose sixth bi...",What is the age or year at which Austrian chil...,of children whose sixth birthday falls before ...,"According to the documents, there are two requ...",What specific requirements must be fulfilled b...,of children whose sixth birthday falls before ...,None,None,None,of children whose sixth birthday falls before ...,[of children whose sixth birthday falls before...
3,3,At what age does compulsory primary education ...,According to Exhibit 1: Levels of Education in...,"Compulsory primary education begins at age 6, ...",['Primary: Compulsory level of education that ...,According to Exhibit 1: Levels of Education in...,At what age does compulsory primary education ...,AZERBAIJAN \nPIRLS 2021 ENCYCLOPEDIA 2 \nThere...,According to Exhibit 1: Levels of Education in...,How long is the compulsory education period in...,AZERBAIJAN \nPIRLS 2021 ENCYCLOPEDIA 2 \nThere...,None,None,None,AZERBAIJAN \nPIRLS 2021 ENCYCLOPEDIA 2 \nThere...,[AZERBAIJAN \nPIRLS 2021 ENCYCLOPEDIA 2 \nTher...
4,4,What purposes do Azerbaijan’s national and sch...,Azerbaijan's national and school-based assessm...,Both national centralized assessments and scho...,['The centralized assessment is funded from th...,"According to the provided text, the primary pu...",What is the primary purpose of Azerbaijan's na...,AZERBAIJAN \nPIRLS 2021 ENCYCLOPEDIA 5 \n \n \...,"According to the provided text, school-based a...",What specific purposes do school-based assessm...,AZERBAIJAN \nPIRLS 2021 ENCYCLOPEDIA 5 \n \n \...,None,None,None,AZERBAIJAN \nPIRLS 2021 ENCYCLOPEDIA 5 \n \n \...,[AZERBAIJAN \nPIRLS 2021 ENCYCLOPEDIA 5 \n \n ...


In [10]:
import asyncio
import nest_asyncio
from ragas import evaluate
from datasets import Dataset
from ragas.run_config import RunConfig

# Ensure data types are consistent for Arrow conversion
# answer_ref caused an ArrowTypeError because it contained integers
eval_dataset_df['answer_ref'] = eval_dataset_df['answer_ref'].astype(str)
eval_dataset_df['user_input'] = eval_dataset_df['user_input'].astype(str)
eval_dataset_df['answer'] = eval_dataset_df['answer'].astype(str)


In [11]:
import copy
from typing import Iterable, List, Optional

# A strict "contract" that we prepend to every Ragas prompt instruction.
STRICT_JSON_CONTRACT = """\
CRITICAL OUTPUT REQUIREMENTS (MUST FOLLOW):
1) Output MUST be a single valid JSON object that conforms EXACTLY to the response schema implied by this prompt.
2) Do NOT wrap the JSON in Markdown. Do NOT use ```json ...``` or backticks.
3) Do NOT add any keys that are not in the schema. Do NOT omit any required keys.
4) Do NOT output any text before or after the JSON. No explanations, no apologies, no commentary.
5) All string values MUST be valid JSON strings (escape newlines as \\n, quotes as \\", etc.).
6) If uncertain, still produce schema-valid JSON with best-effort values.
"""

def override_metric_prompts_to_force_json(metric) -> None:
    """
    Harden ALL prompts inside a Ragas metric to strongly enforce schema-valid JSON output.

    Works for ContextPrecision, ContextRecall, Faithfulness, AnswerCorrectness, and most other
    PromptMixin-based metrics because it relies on get_prompts()/set_prompts() rather than
    internal attribute names.
    """
    # Ragas PromptMixin provides get_prompts() and set_prompts().  :contentReference[oaicite:1]{index=1}
    if not hasattr(metric, "get_prompts") or not hasattr(metric, "set_prompts"):
        raise TypeError(
            f"{type(metric).__name__} does not expose get_prompts()/set_prompts(). "
            "Ensure you are passing an initialized Ragas metric object."
        )

    current = metric.get_prompts()  # dict: {prompt_name: PydanticPrompt}
    patched = {}

    for prompt_name, prompt in current.items():
        p = copy.deepcopy(prompt)

        # Prepend strict contract to the existing instruction.
        # (This is the highest-leverage change for models that otherwise add prose / code fences.)
        p.instruction = STRICT_JSON_CONTRACT + "\n\n" + (p.instruction or "")

        # Optionally: increase the chance the model obeys by making the schema requirement explicit.
        # Many Ragas prompts already include schema guidance; this reinforces it.
        p.instruction += "\n\nREMINDER: Return ONLY the JSON object, nothing else."

        patched[prompt_name] = p

    # Replace prompts in the metric (by prompt name, not by internal variable name).
    metric.set_prompts(**patched)

def override_prompts_for_metrics(metrics: Iterable) -> None:
    """
    Apply the strict JSON prompt override to a list/iterable of metric objects.
    """
    for m in metrics:
        override_metric_prompts_to_force_json(m)

In [ ]:
import nest_asyncio
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import ContextPrecision, ContextRecall, Faithfulness, AnswerCorrectness
from ragas.run_config import RunConfig
nest_asyncio.apply()

# eval_dataset = Dataset.from_pandas(qa_data)
eval_dataset = Dataset.from_pandas(eval_dataset_df.rename(columns={'answer': 'response'}))

run_config = RunConfig(max_workers=1, timeout=600)

metrics = [
    ContextPrecision(llm=llm),
    ContextRecall(llm=llm),
    Faithfulness(llm=llm),
    AnswerCorrectness(llm=llm),
]

override_prompts_for_metrics(metrics)

results = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    llm=llm,
    embeddings=embeddings,
    raise_exceptions=False,
    run_config=run_config,
)

eval_dataset_df["context_precision_score"] = results["context_precision"]
eval_dataset_df["context_recall_score"] = results["context_recall"]
eval_dataset_df["faithfulness_score"] = results["faithfulness"]
eval_dataset_df["answer_correctness_score"] = results["answer_correctness"]
display(eval_dataset_df[["context_precision_score", "context_recall_score", "faithfulness_score", "answer_correctness_score"]].head())

/tmp/ipython-input-228115929.py:4: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import ContextPrecision, ContextRecall, Faithfulness, AnswerCorrectness
/tmp/ipython-input-228115929.py:4: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import ContextPrecision, ContextRecall, Faithfulness, AnswerCorrectness
/tmp/ipython-input-228115929.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import ContextPrecision, ContextR

Evaluating:   0%|          | 0/796 [00:00<?, ?it/s]

In [ ]:
# Save result to CSV
eval_dataset_df.to_csv('/content/drive/MyDrive/RAG/results/eval_adv_with_llama_20260108.csv', index=False)